# Test — Logování v Pythonu

**Pravidla:**
- Používejte pouze modul `logging` (případně `logging.handlers`).
- Nepoužívejte `print()` místo logování (kromě výslovně zadaných případů).
- Zprávy v logu piště anglicky, pokud zadání neříká jinak.
- Před každým úkolem můžete restartovat kernel (`basicConfig` funguje jen jednou). Případně pracujte v samostatných souborech .py.

| Část | Body |
|---|---|
| A — Teorie | 10 |
| B — Praktické úlohy | 40 |
| **Celkem** | **50** |

Hodnocení: 45–50 = 1, 38–44 = 2, 30–37 = 3, 25–29 = 4, méně = 5

---
# Část A — Teorie (10 b)

Odpovězte do buněk pod otázkami (1 bod za otázku).

**A1.** Uveďte alespoň tři důvody, proč se aplikace logují.

*Odpověď:*
- **Ladění a hledání chyb** – log ukáže, co se dělo těsně před chybou (i na produkci, kde nelze použít debugger).
- **Monitorování provozu** – sledování stavu aplikace, výkonu, upozornění na problémy (alerting).
- **Audit a bezpečnost** – záznam, kdo co kdy udělal (přihlášení, změny dat), dohledání incidentů.
- **Analýza a statistiky** – počty požadavků, doby odezvy, chování uživatelů.

**A2.** Popište zodpovědnost čtyř hlavních částí logovacího procesu: Logger, Handler, Formatter, Filter.

*Odpověď:*
- **Logger** – vstupní bod, přes který aplikace posílá zprávy (`logger.info(...)`). Má jméno (hierarchie), úroveň a vytváří z volání objekt `LogRecord`.
- **Handler** – určuje, **kam** se záznam pošle (konzole, soubor, síť, e-mail…). Jeden logger může mít více handlerů, každý s vlastní úrovní.
- **Formatter** – určuje, **jak** záznam vypadá jako text (čas, úroveň, jméno loggeru, zpráva…).
- **Filter** – jemnější rozhodnutí, **zda** záznam projde (např. podle jména loggeru nebo obsahu zprávy); může záznam i upravit.

**A3.** Seřaďte úrovně závažnosti od nejnižší po nejvyšší a uveďte jejich číselné hodnoty.

*Odpověď:* `NOTSET` (0) < `DEBUG` (10) < `INFO` (20) < `WARNING` (30) < `ERROR` (40) < `CRITICAL` (50)

**A4.** Logger má nastavenou úroveň `WARNING`. Které z volání `debug`, `info`, `warning`, `error`, `critical` se zobrazí?

*Odpověď:* Zobrazí se `warning`, `error` a `critical` (úroveň ≥ 30). `debug` a `info` se zahodí.

**A5.** Jaká je výchozí úroveň loggeru bez volání `basicConfig`? Co je specifické na `basicConfig()` při opakovaném volání?

*Odpověď:* Výchozí úroveň root loggeru je `WARNING` (nové pojmenované loggery mají `NOTSET` a úroveň dědí od rodiče, tedy efektivně také `WARNING`).
`basicConfig()` nastaví root logger jen tehdy, pokud ještě nemá žádné handlery – opakované volání tedy **nic neudělá** (pokud nepoužijeme parametr `force=True`, který stávající handlery odstraní a nakonfiguruje root znovu).

**A6.** Co znamená, že loggery tvoří hierarchii? Co se stane se zprávou z loggeru `app.database`, který nemá žádný handler, ale root logger handler má?

*Odpověď:* Loggery se uspořádávají podle jmen oddělených tečkou: `app.database` je potomkem `app`, ten je potomkem root loggeru. Potomci dědí úroveň (pokud mají `NOTSET`) a zprávy se **propagují** (`propagate=True`) nahoru k rodičům.
Zpráva z `app.database` bez vlastního handleru proto projde (pokud ji pustí úroveň loggeru) až k root loggeru a vypíše ji handler root loggeru.

**A7.** Jaký je rozdíl mezi úrovní nastavenou na loggeru a úrovní nastavenou na handleru?

*Odpověď:* Úroveň **loggeru** je první síto – zprávy pod ní se vůbec nevytvoří a nedostanou se k žádnému handleru. Úroveň **handleru** se uplatní až potom a platí jen pro daný výstup. Díky tomu může logger propouštět např. od `DEBUG`, konzolový handler zobrazovat od `INFO` a souborový ukládat jen od `WARNING`. Zpráva se objeví jen tehdy, pokud projde oběma úrovněmi.

**A8.** K čemu slouží `RotatingFileHandler` a které dva parametry jsou pro něj klíčové?

*Odpověď:* `RotatingFileHandler` zapisuje do souboru, a když soubor dosáhne určité velikosti, přejmenuje ho (`app.log` → `app.log.1` …) a začne psát do nového. Tím zabraňuje neomezenému růstu logu.
Klíčové parametry: **`maxBytes`** (maximální velikost souboru, po jejím dosažení proběhne rotace) a **`backupCount`** (kolik starých záloh se uchovává; nejstarší se mažou).

**A9.** Jak zalogujete výjimku včetně tracebacku? Uveďte dva způsoby.

*Odpověď:*
1. V bloku `except` zavolat `logger.exception("zpráva")` – loguje na úrovni `ERROR` a automaticky připojí traceback.
2. Použít libovolnou metodu s parametrem `exc_info=True`, např. `logger.error("zpráva", exc_info=True)` nebo `logger.warning(..., exc_info=True)`.

**A10.** Uveďte alespoň čtyři výhody `logging` oproti `print()` a dvě pravidla pro psaní dobrých logů (např. co nelogovat).

*Odpověď:*
**Výhody `logging` oproti `print()`:**
- úrovně závažnosti a možnost filtrovat, co se zobrazí (bez mazání kódu),
- více výstupů najednou (konzole, soubor, síť) s různými úrovněmi,
- automatické informace v záznamu (čas, modul, řádek, jméno loggeru) přes formát,
- rotace souborů, logování výjimek s tracebackem,
- konfigurace na jednom místě; knihovny mohou logovat a aplikace rozhodne, co s tím.

**Pravidla pro dobré logy:**
- nelogovat citlivé údaje (hesla, tokeny, čísla karet, osobní údaje),
- volit správnou úroveň a psát konkrétní zprávy s kontextem (co, s jakými hodnotami), ne jen „Error“,
- nelogovat zbytečně mnoho (např. v těsných smyčkách) – log musí zůstat čitelný.

---
# Část B — Praktické úlohy (40 b)

## Úloha 1 — Základní logger (5 b)

1. Vytvořte logger `"test1"` s úrovní `DEBUG`.
2. Přidejte `StreamHandler` s formátem `ÚROVEŇ | zpráva` (úroveň zarovnaná na 8 znaků).
3. Vypište po jedné zprávě na každé úrovni.

<details>
<summary>🔎 Očekávaný výstup</summary>

```
DEBUG    | debug message
INFO     | info message
WARNING  | warning message
ERROR    | error message
CRITICAL | critical message
```

</details>

In [ ]:
import logging

logger = logging.getLogger("test1")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()        # proti duplicitám při opakovaném spuštění buňky
logger.propagate = False

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))
logger.addHandler(handler)

logger.debug("debug message")
logger.info("info message")
logger.warning("warning message")
logger.error("error message")
logger.critical("critical message")

## Úloha 2 — Konzole + soubor s různými úrovněmi (6 b)

1. Logger `"test2"` s úrovní `DEBUG`.
2. Do konzole vypisujte od úrovně `INFO`, zpráva se vypíše jako `úroveň: zpráva`.
3. Od úrovně `WARNING` ukládejte log do souboru `test2.log` (soubor se při každém spuštění přepíše), řádek v souboru: `čas - název loggeru - úroveň - zpráva`.
4. Zalogujte DEBUG, INFO, WARNING, ERROR.
5. Na konci vypište obsah souboru pomocí `open()`.

<details>
<summary>🔎 Očekávaný výstup</summary>

```
INFO: Info message
WARNING: Warning message
ERROR: Error message

--- test2.log ---
2026-10-12 10:00:00,000 - test2 - WARNING - Warning message
2026-10-12 10:00:00,001 - test2 - ERROR - Error message
```

</details>

In [ ]:
import logging

logger = logging.getLogger("test2")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
logger.propagate = False

console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

file_handler = logging.FileHandler("test2.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.WARNING)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))

logger.addHandler(console)
logger.addHandler(file_handler)

logger.debug("Debug message")
logger.info("Info message")
logger.warning("Warning message")
logger.error("Error message")

file_handler.close()
print("\n--- test2.log ---")
with open("test2.log", encoding="utf-8") as f:
    print(f.read())

## Úloha 3 — Hierarchie loggerů a filtr (6 b)

1. Na root logger přidejte `StreamHandler` (výpis ve tvaru `název loggeru - zpráva`) a nastavte root na `DEBUG`.
2. Přidejte filtr, který propustí jen zprávy z loggeru `shop.orders` (a jeho podřízených loggerů).
3. Zalogujte po jedné INFO zprávě z loggerů: `shop`, `shop.orders`, `shop.orders.items`, `shop.users`.
4. Pod buňku do komentáře napište, které zprávy se zobrazily a proč.

<details>
<summary>🔎 Očekávaný výstup</summary>

```
shop.orders - Order created
shop.orders.items - Item added
```

</details>

In [ ]:
import logging

root = logging.getLogger()
root.setLevel(logging.DEBUG)
root.handlers.clear()

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(name)s - %(message)s"))
# Filtr musí být na handleru – filtr na loggeru se na propagované zprávy z potomků neuplatní.
handler.addFilter(logging.Filter("shop.orders"))
root.addHandler(handler)

logging.getLogger("shop").info("Shop opened")
logging.getLogger("shop.orders").info("Order created")
logging.getLogger("shop.orders.items").info("Item added")
logging.getLogger("shop.users").info("User registered")

# Zobrazily se jen zprávy z "shop.orders" a "shop.orders.items".
# Všechny čtyři loggery nemají vlastní handler, zprávy se propagují až k root loggeru
# a jeho handleru. logging.Filter("shop.orders") propustí záznam jen tehdy, když jméno
# loggeru je přesně "shop.orders" nebo začíná "shop.orders." (potomek).
# "shop" je rodič (ne potomek) a "shop.users" je jiná větev, proto byly zahozeny.

**Vysvětlení (Úloha 3):** Zobrazily se jen zprávy `shop.orders - Order created` a `shop.orders.items - Item added`. Žádný z loggerů `shop.*` nemá vlastní handler, zprávy se propagují k root loggeru. Filtr `logging.Filter("shop.orders")` na handleru propustí jen záznamy, jejichž jméno loggeru je `shop.orders` nebo začíná `shop.orders.` – tedy logger samotný a jeho potomky. `shop` je rodič a `shop.users` sourozenecká větev, proto byly odfiltrovány.

## Úloha 4 — Vlastní filtr jako funkce (5 b)

Vytvořte logger `"test4"` se `StreamHandler` (výpis ve tvaru `úroveň: zpráva`). Napište funkci `no_secrets(record)`, která zahodí každou zprávu obsahující slovo `password` (bez ohledu na velikost písmen). Přidejte ji jako filtr na handler.

Zalogujte:
- `"User logged in"`
- `"Password is 1234"`
- `"Request finished"`

<details>
<summary>🔎 Očekávaný výstup</summary>

```
INFO: User logged in
INFO: Request finished
```

</details>

In [ ]:
import logging

logger = logging.getLogger("test4")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
logger.propagate = False


def no_secrets(record):
    """Zahodí záznam, pokud zpráva obsahuje slovo 'password' (bez ohledu na velikost písmen)."""
    return "password" not in record.getMessage().lower()


handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
handler.addFilter(no_secrets)
logger.addHandler(handler)

logger.info("User logged in")
logger.info("Password is 1234")
logger.info("Request finished")

## Úloha 5 — Logování výjimek (6 b)

Funkce `bezpecne_deleni` je už napsaná, ale nic neloguje. Vytvořte logger `"test5"` se `StreamHandler` a do funkce doplňte logování:

- při úspěchu INFO `"a / b = výsledek"`,
- při `ZeroDivisionError` chybu **včetně tracebacku** (`Division by zero: 5 / 0`),
- při `TypeError` `WARNING` (`Invalid types: 'a' and 2`) s tracebackem pomocí `exc_info=True`.

<details>
<summary>🔎 Očekávaný výstup</summary>

```
10 / 4 = 2.5
Division by zero: 5 / 0
Traceback (most recent call last):
  ...
ZeroDivisionError: division by zero
Invalid types: 'a' and 2
Traceback (most recent call last):
  ...
TypeError: unsupported operand type(s) for /: 'str' and 'int'
```

</details>

In [ ]:
import logging

logger = logging.getLogger("test5")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
logger.propagate = False

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(handler)


def bezpecne_deleni(a, b):
    try:
        vysledek = a / b
        logger.info(f"{a} / {b} = {vysledek}")
        return vysledek
    except ZeroDivisionError:
        logger.exception(f"Division by zero: {a} / {b}")   # ERROR + traceback
        return None
    except TypeError:
        logger.warning(f"Invalid types: {a!r} and {b!r}", exc_info=True)
        return None


bezpecne_deleni(10, 4)
bezpecne_deleni(5, 0)
bezpecne_deleni("a", 2)

## Úloha 6 — Rotace logů (5 b)

1. Logger `"test6"` s úrovní `INFO`.
2. `RotatingFileHandler` do `test6.log`, `maxBytes=2000`, `backupCount=2`, `encoding="utf-8"`.
3. Zapište 100 zpráv `"Message number i"`.
4. Pomocí `os.listdir` / `glob` vypište soubory začínající na `test6.log`.
5. Do komentáře napište, proč jich není víc než 3.

<details>
<summary>🔎 Očekávaný výstup</summary>

```
['test6.log', 'test6.log.1', 'test6.log.2']
```

</details>

In [ ]:
import logging
from logging.handlers import RotatingFileHandler
import glob
import os

# úklid po předchozím spuštění
for f in glob.glob("test6.log*"):
    os.remove(f)

logger = logging.getLogger("test6")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False

handler = RotatingFileHandler("test6.log", maxBytes=2000, backupCount=2, encoding="utf-8")
handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(handler)

for i in range(100):
    logger.info(f"Message number {i}")

handler.close()
print(sorted(glob.glob("test6.log*")))

# Proč nejsou víc než 3 soubory:
# Vždy existuje jeden aktivní soubor test6.log a nejvýše backupCount=2 záloh.
# Při každé rotaci se test6.log.1 přejmenuje na test6.log.2, test6.log na test6.log.1
# a nejstarší záloha (dosavadní test6.log.2) se smaže. Starší zprávy se tedy ztratí,
# celkem jsou maximálně 1 + 2 = 3 soubory.

## Úloha 7 — Komplexní úloha: třída `Sklad` (7 b)

Třída `Sklad` i testovací scénář jsou hotové, ale nic neloguje. Nakonfigurujte logování a vložte logger do metod.

**Konfigurace:**
- logger `"sklad"`, úroveň `DEBUG`,
- `StreamHandler` (`DEBUG`, výpis `úroveň: zpráva`) a `FileHandler` `sklad.log` (`INFO`, `mode="w"`, řádek v souboru `čas - úroveň - zpráva`).

**Logování v metodách:**
- `pridat` — INFO `Added X pcs of NAME`, DEBUG se změnou stavu `NAME: old -> new`,
- `odebrat` — neexistující položka: `ERROR`; nedostatek kusů: `WARNING` (nic se neodebere); jinak INFO + DEBUG jako u přidání.

<details>
<summary>🔎 Očekávaný výstup (konzole)</summary>

```
INFO: Added 10 pcs of apple
DEBUG: apple: 0 -> 10
INFO: Removed 3 pcs of apple
DEBUG: apple: 10 -> 7
WARNING: Not enough apple (have 7, requested 20)
ERROR: Item pear does not exist
```

V souboru `sklad.log` budou jen řádky INFO, WARNING a ERROR.

</details>

In [ ]:
import logging

logger = logging.getLogger("sklad")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
logger.propagate = False

console = logging.StreamHandler()
console.setLevel(logging.DEBUG)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

file_handler = logging.FileHandler("sklad.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))

logger.addHandler(console)
logger.addHandler(file_handler)


class Sklad:
    def __init__(self):
        self.polozky = {}

    def pridat(self, nazev, kusu):
        stary = self.polozky.get(nazev, 0)
        self.polozky[nazev] = stary + kusu
        logger.info(f"Added {kusu} pcs of {nazev}")
        logger.debug(f"{nazev}: {stary} -> {self.polozky[nazev]}")

    def odebrat(self, nazev, kusu):
        if nazev not in self.polozky:
            logger.error(f"Item {nazev} does not exist")
            return
        stary = self.polozky[nazev]
        if stary < kusu:
            logger.warning(f"Not enough {nazev} (have {stary}, requested {kusu})")
            return
        self.polozky[nazev] = stary - kusu
        logger.info(f"Removed {kusu} pcs of {nazev}")
        logger.debug(f"{nazev}: {stary} -> {self.polozky[nazev]}")


# scénář (hotovo)
sklad = Sklad()
sklad.pridat("apple", 10)
sklad.odebrat("apple", 3)
sklad.odebrat("apple", 20)
sklad.odebrat("pear", 1)

print("\n--- sklad.log ---")
print(open("sklad.log", encoding="utf-8").read())